<a href="https://colab.research.google.com/github/jaredaalarcon-dev/ENTREGABLE/blob/main/Laboratorio06_Spark_Sentimiento_Viralidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 06 — Apache Spark + CRISP-ML
## Análisis de sentimiento y predicción de viralidad en campañas de marketing

**Carrera:** Ingeniería de Software con IA
**Instructor:** Remigio Huarcaya Almeyda
**Dataset:** `dataset_marketing_viral.xlsx` (Caso de estudio: *Previsión de Éxito en Campañas de Marketing Viral*, Anexo del Laboratorio 06)

---

### Nota sobre el enfoque

La consigna original de la Actividad 2 pide un análisis de **sentimiento** (positivo / negativo / neutro) de comentarios sobre una institución educativa. El dataset que se nos entregó, sin embargo, corresponde al caso de estudio del Anexo — publicaciones sobre el lanzamiento de un producto tecnológico (*TecnoNova X1*), con una columna `target` (1 = viral, 0 = no viral).

Para no perder ninguno de los dos objetivos del laboratorio, este cuaderno desarrolla **ambas cosas** siguiendo la metodología **CRISP-ML(Q)**:

1. Un módulo de **análisis de sentimiento** sobre el texto de las publicaciones (adaptando el objetivo "comentarios sobre institución" a "comentarios sobre marca/producto", que es la naturaleza real de los datos).
2. Un módulo de **modelo predictivo de viralidad** (el reto explícito del caso de estudio), usando Spark MLlib, donde el sentimiento detectado se incorpora como una variable (*feature*) más del modelo.

Esta combinación es coherente porque, como vamos a comprobar en el EDA, la intensidad emocional del texto (muy positivo o muy negativo) está relacionada con qué tan viral se vuelve una publicación, mientras que los posts neutros/informativos tienden a no viralizarse.


## Fase 1 — CRISP-ML(Q): Comprensión del negocio (Business Understanding)

**Contexto:** TecnoNova, una marca de tecnología, va a lanzar el producto *TecnoNova X1*. Cuenta con un histórico de publicaciones en redes sociales que mencionan la marca. El volumen de datos hace inviable el análisis con herramientas tradicionales (Excel, Python en memoria local), por lo que se decide usar **Apache Spark**.

**Objetivo de negocio:** anticipar, antes de publicar un mensaje, si tendrá **alto impacto (viral)** o **bajo impacto (ignorado)**, y entender qué papel juega el sentimiento del texto en ese resultado.

**Objetivo de minería de datos:**
- Clasificar el sentimiento (positivo / negativo / neutro) de cada publicación.
- Entrenar un modelo de clasificación binaria (`target`: 1 = viral, 0 = no viral) usando como variables predictoras: el texto (vectorizado), el sentimiento derivado, la hora de publicación, el número de seguidores del autor y el tipo de contenido multimedia.

**Criterio de éxito:** un modelo con un AUC y una exactitud (accuracy) sensiblemente mejores que un clasificador aleatorio (0.5), que además sea interpretable para el equipo de marketing.


## Configuración del entorno

Montamos Google Drive (donde subiste `dataset_marketing_viral.xlsx`) e instalamos PySpark.

> ⚠️ Ajusta la variable `RUTA_ARCHIVO` a la ruta real dentro de tu Drive donde guardaste el archivo.


In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Instalar PySpark y una librería para leer archivos .xlsx dentro de Spark/Pandas
!pip install pyspark openpyxl -q

In [ ]:
# Ruta al dataset dentro de tu Google Drive
RUTA_ARCHIVO = '/content/drive/MyDrive/colaba/dataset_marketing_viral.xlsx'

## Creación de la SparkSession

Como vimos en el Laboratorio 06, el `Driver` necesita una `SparkSession` para poder analizar el código, construir el DAG y repartir el trabajo entre los `Worker Nodes` a través del `SparkContext`.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TecnoNova - Analisis de Sentimiento y Viralidad") \
    .getOrCreate()

sc = spark.sparkContext
spark

## Fase 1 (cont.) — Comprensión de los datos (Data Understanding)

Spark no lee `.xlsx` de forma nativa (solo CSV, JSON, Parquet, etc.), así que primero cargamos el Excel con `pandas` y luego lo convertimos a un DataFrame distribuido de Spark. Esto es habitual cuando el archivo fuente es pequeño (miles de filas) pero el procesamiento posterior (texto, features, modelo) se quiere hacer de forma distribuida.


In [ ]:
import pandas as pd

# 1) Leer el Excel con pandas
pdf = pd.read_excel(RUTA_ARCHIVO)
print("Filas y columnas (pandas):", pdf.shape)
pdf.head()

Filas y columnas (pandas): (400, 5)


,texto_post,hora_publicacion,num_seguidores_autor,contenido_multimedia,target
0,¿Alguien ya probó el TecnoNova X1 de TecnoNova...,23,12217,Video,0
1,"TecnoNova lanza el TecnoNova X1 este mes, aquí...",18,32549,Video,0
2,El servicio postventa de TecnoNova con el Tecn...,19,31415,Video,1
3,"El diseño del TecnoNova X1 es hermoso, TecnoNo...",8,17762,Solo texto,0
4,TecnoNova anunció actualizaciones de software ...,10,11966,Solo texto,0


In [ ]:
# 2) Convertir el pandas DataFrame en un Spark DataFrame
data = spark.createDataFrame(pdf)
data.printSchema()

root
 |-- texto_post: string (nullable = true)
 |-- hora_publicacion: long (nullable = true)
 |-- num_seguidores_autor: long (nullable = true)
 |-- contenido_multimedia: string (nullable = true)
 |-- target: long (nullable = true)



In [ ]:
data.show(5, truncate=60)

+------------------------------------------------------------+----------------+--------------------+--------------------+------+
|                                                  texto_post|hora_publicacion|num_seguidores_autor|contenido_multimedia|target|
+------------------------------------------------------------+----------------+--------------------+--------------------+------+
|¿Alguien ya probó el TecnoNova X1 de TecnoNova? Estoy pen...|              23|               12217|               Video|     0|
|TecnoNova lanza el TecnoNova X1 este mes, aquí les dejo l...|              18|               32549|               Video|     0|
|El servicio postventa de TecnoNova con el TecnoNova X1 fu...|              19|               31415|               Video|     1|
|El diseño del TecnoNova X1 es hermoso, TecnoNova pensó en...|               8|               17762|          Solo texto|     0|
|TecnoNova anunció actualizaciones de software para el Tec...|              10|               119

In [ ]:
# Cantidad total de registros
print("Total de registros:", data.count())

Total de registros: 400


In [ ]:
# Distribución de la variable objetivo (target)
data.groupBy('target').count().show()

+------+-----+
|target|count|
+------+-----+
|     0|  264|
|     1|  136|
+------+-----+



In [ ]:
# Distribución del tipo de contenido multimedia
data.groupBy('contenido_multimedia').count().orderBy('count', ascending=False).show()

+--------------------+-----+
|contenido_multimedia|count|
+--------------------+-----+
|               Video|  173|
|              Imagen|  142|
|          Solo texto|   85|
+--------------------+-----+



In [ ]:
# Cruce: contenido multimedia vs viralidad
data.groupBy('contenido_multimedia', 'target').count().orderBy('contenido_multimedia').show()

+--------------------+------+-----+
|contenido_multimedia|target|count|
+--------------------+------+-----+
|              Imagen|     1|   34|
|              Imagen|     0|  108|
|          Solo texto|     0|   75|
|          Solo texto|     1|   10|
|               Video|     1|   92|
|               Video|     0|   81|
+--------------------+------+-----+



In [ ]:
from pyspark.sql.functions import col, avg

# ¿Los posts virales tienen en promedio más seguidores o se publican a otra hora?
data.groupBy('target').agg(
    avg('num_seguidores_autor').alias('promedio_seguidores'),
    avg('hora_publicacion').alias('hora_promedio')
).show()

+------+-------------------+------------------+
|target|promedio_seguidores|     hora_promedio|
+------+-------------------+------------------+
|     0| 15436.117424242424| 11.06439393939394|
|     1| 27926.014705882353|12.786764705882353|
+------+-------------------+------------------+



## Fase 2 — Preparación de los datos


### 2.1 Análisis de sentimiento


Para clasificar cada publicación como **positiva**, **negativa** o **neutra** usamos un enfoque de **lexicón**: listas de palabras asociadas a sentimiento positivo y negativo. Contamos cuántas palabras de cada lista aparecen en el texto y decidimos la etiqueta según cuál predomina.

Este enfoque es simple pero perfectamente válido para un caso académico y tiene la ventaja de ser 100% explicable (a diferencia de una caja negra), algo valioso cuando se necesita justificar decisiones de negocio.


In [ ]:
# Lexicón básico de sentimiento en español para el dominio de reseñas de producto/servicio
palabras_positivas = [
    "impecable", "recomendado", "hermoso", "excelente", "genial", "increíble",
    "encanta", "encantó", "perfecto", "rápido", "satisfecho", "feliz",
    "buena", "bueno", "mejor", "fantástico", "sorprendente", "agradable",
    "cómodo", "eficiente", "vale la pena", "me encanta"
]

palabras_negativas = [
    "pésimo", "pésima", "mala", "malo", "terrible", "decepción", "decepcionante",
    "traba", "trabajo", "lento", "falla", "falló", "error", "problema",
    "esperando", "queja", "reclamo", "defectuoso", "no funciona",
    "estafa", "horrible", "molesto", "frustrante"
]

print("Palabras positivas:", len(palabras_positivas))
print("Palabras negativas:", len(palabras_negativas))

Palabras positivas: 22
Palabras negativas: 23


In [ ]:
from pyspark.sql.functions import udf, lower
from pyspark.sql.types import StringType

def clasificar_sentimiento(texto):
    """Cuenta coincidencias de palabras positivas y negativas en el texto
    y devuelve la etiqueta de sentimiento predominante."""
    if texto is None:
        return "neutro"
    texto_low = texto.lower()
    pos = sum(1 for palabra in palabras_positivas if palabra in texto_low)
    neg = sum(1 for palabra in palabras_negativas if palabra in texto_low)
    if pos > neg:
        return "positivo"
    elif neg > pos:
        return "negativo"
    else:
        return "neutro"

sentimiento_udf = udf(clasificar_sentimiento, StringType())

data = data.withColumn("sentimiento", sentimiento_udf(col("texto_post")))
data.select("texto_post", "sentimiento").show(10, truncate=60)

+------------------------------------------------------------+-----------+
|                                                  texto_post|sentimiento|
+------------------------------------------------------------+-----------+
|¿Alguien ya probó el TecnoNova X1 de TecnoNova? Estoy pen...|     neutro|
|TecnoNova lanza el TecnoNova X1 este mes, aquí les dejo l...|     neutro|
|El servicio postventa de TecnoNova con el TecnoNova X1 fu...|   positivo|
|El diseño del TecnoNova X1 es hermoso, TecnoNova pensó en...|   positivo|
|TecnoNova anunció actualizaciones de software para el Tec...|     neutro|
|Pésimo servicio al cliente de TecnoNova, llevo un mes esp...|   negativo|
|Tutorial: cómo configurar el TecnoNova X1 de TecnoNova pa...|     neutro|
|Reseña técnica: el TecnoNova X1 de TecnoNova incluye proc...|     neutro|
|El TecnoNova X1 estará disponible en tiendas físicas de T...|     neutro|
|Comparando el TecnoNova X1 con la competencia, ambos tien...|     neutro|
+------------------------

In [ ]:
# Distribución de sentimientos detectados
data.groupBy("sentimiento").count().orderBy("count", ascending=False).show()

+-----------+-----+
|sentimiento|count|
+-----------+-----+
|     neutro|  287|
|   positivo|   57|
|   negativo|   56|
+-----------+-----+



In [ ]:
# ¿El sentimiento se relaciona con la viralidad? (idea central del caso de negocio)
data.groupBy("sentimiento", "target").count().orderBy("sentimiento").show()

+-----------+------+-----+
|sentimiento|target|count|
+-----------+------+-----+
|   negativo|     0|   39|
|   negativo|     1|   17|
|     neutro|     1|   87|
|     neutro|     0|  200|
|   positivo|     1|   32|
|   positivo|     0|   25|
+-----------+------+-----+



> **Lectura de negocio:** si el cruce anterior muestra que los posts **positivos** y **negativos** (contenido con carga emocional) tienen una proporción de `target = 1` (viral) más alta que los posts **neutros**, confirmamos la hipótesis de que la intensidad emocional —no necesariamente lo "bueno" o "malo" del comentario— es lo que impulsa la viralidad. Esto es consistente con la observación inicial de los datos: reseñas técnicas/informativas (neutras) casi no se vuelven virales.


### 2.2 Ingeniería de características (Feature Engineering) para el modelo predictivo

Construimos un pipeline de Spark MLlib que combina:

- **Texto** (`texto_post`): tokenización → eliminación de stopwords → TF-IDF.
- **Sentimiento** (`sentimiento`): variable categórica → índice → one-hot encoding.
- **Contenido multimedia** (`contenido_multimedia`): variable categórica → índice → one-hot encoding.
- **Variables numéricas**: `hora_publicacion`, `num_seguidores_autor`.

Todas estas features se combinan en un único vector con `VectorAssembler`, que es lo que finalmente recibe el algoritmo de clasificación.


In [ ]:
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF,
    StringIndexer, OneHotEncoder, VectorAssembler
)

# --- Texto ---
tokenizer = Tokenizer(inputCol="texto_post", outputCol="palabras")

# Stopwords en español (PySpark trae una lista base para "spanish")
remover = StopWordsRemover(inputCol="palabras", outputCol="palabras_filtradas")
remover.setStopWords(StopWordsRemover.loadDefaultStopWords("spanish"))

hashingTF = HashingTF(inputCol="palabras_filtradas", outputCol="tf_features", numFeatures=2**12)
idf = IDF(inputCol="tf_features", outputCol="tfidf_features")

# --- Categóricas ---
sentimiento_indexer = StringIndexer(inputCol="sentimiento", outputCol="sentimiento_idx")
sentimiento_ohe = OneHotEncoder(inputCol="sentimiento_idx", outputCol="sentimiento_vec")

media_indexer = StringIndexer(inputCol="contenido_multimedia", outputCol="media_idx")
media_ohe = OneHotEncoder(inputCol="media_idx", outputCol="media_vec")

# --- Ensamblado final ---
assembler = VectorAssembler(
    inputCols=["tfidf_features", "sentimiento_vec", "media_vec",
               "hora_publicacion", "num_seguidores_autor"],
    outputCol="features"
)

## Fase 3 — Modelado (Modeling)

Usamos **Regresión Logística** de `pyspark.ml.classification`: es un algoritmo simple, rápido de entrenar sobre Spark y fácil de interpretar (los coeficientes indican qué variables aumentan o disminuyen la probabilidad de viralidad), lo cual es importante para poder explicarle el resultado al equipo de marketing.

Armamos todo el flujo (features + modelo) en un `Pipeline` de MLlib, y dividimos los datos en 80% entrenamiento / 20% prueba.


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

lr = LogisticRegression(featuresCol="features", labelCol="target")

pipeline = Pipeline(stages=[
    tokenizer, remover, hashingTF, idf,
    sentimiento_indexer, sentimiento_ohe,
    media_indexer, media_ohe,
    assembler, lr
])

In [ ]:
# División train / test (80/20), con semilla fija para reproducibilidad
train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)

print("Registros de entrenamiento:", train_df.count())
print("Registros de prueba:", test_df.count())

Registros de entrenamiento: 332
Registros de prueba: 68


In [ ]:
# Entrenamiento del pipeline completo
modelo = pipeline.fit(train_df)
print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


In [ ]:
# Predicciones sobre el conjunto de prueba
predicciones = modelo.transform(test_df)
predicciones.select("texto_post", "target", "sentimiento", "prediction", "probability").show(10, truncate=50)

+--------------------------------------------------+------+-----------+----------+------------------------------------------+
|                                        texto_post|target|sentimiento|prediction|                               probability|
+--------------------------------------------------+------+-----------+----------+------------------------------------------+
|Cancelé mi pedido del TecnoNova X1, TecnoNova t...|     0|     neutro|       1.0|   [0.2500412177554413,0.7499587822445587]|
|Comparando el TecnoNova X1 con la competencia, ...|     0|     neutro|       0.0|  [0.9426099710767615,0.05739002892323852]|
|Comparando el TecnoNova X1 con la competencia, ...|     0|     neutro|       0.0|  [0.9036008359354892,0.09639916406451077]|
|Comparando el TecnoNova X1 con la competencia, ...|     1|     neutro|       1.0|   [0.2969845868833596,0.7030154131166404]|
|Decepcionado con el TecnoNova X1, esperaba much...|     0|     neutro|       0.0| [0.9999999300439311,6.9956068937671

## Fase 4 — Evaluación (Evaluation)

Medimos el desempeño del modelo con:

- **AUC (Área bajo la curva ROC)**: qué tan bien separa el modelo la clase viral de la no viral.
- **Exactitud (Accuracy)**.
- **Matriz de confusión** para ver los errores en detalle (falsos positivos vs falsos negativos).


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

evaluator_auc = BinaryClassificationEvaluator(labelCol="target", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predicciones)
print(f"AUC (área bajo la curva ROC): {auc:.3f}")

evaluator_acc = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predicciones)
print(f"Exactitud (accuracy): {accuracy:.3f}")

AUC (área bajo la curva ROC): 0.834
Exactitud (accuracy): 0.765


In [ ]:
# Matriz de confusión
predicciones.groupBy("target", "prediction").count().orderBy("target", "prediction").show()

+------+----------+-----+
|target|prediction|count|
+------+----------+-----+
|     0|       0.0|   33|
|     0|       1.0|    7|
|     1|       0.0|    9|
|     1|       1.0|   19|
+------+----------+-----+



> **Interpretación:** un AUC cercano a 1.0 indica que el modelo distingue bien entre publicaciones virales y no virales; un AUC cercano a 0.5 indicaría que el modelo no aporta más que una decisión al azar. Dado que este es un dataset sintético de tamaño reducido (400 registros), es normal que las métricas varíen si se cambia la semilla de la partición train/test — en un escenario real (millones de publicaciones, como se plantea en el caso de negocio) se recomienda validación cruzada y un conjunto de prueba mucho más grande.


## Fase 5 — Despliegue (Deployment)

En un escenario productivo, este pipeline se integraría como un servicio que el equipo de marketing consulta **antes** de publicar un mensaje:

1. El community manager redacta el post.
2. El pipeline de Spark (guardado en disco) calcula el sentimiento y las features, y el modelo entrega una probabilidad de viralidad.
3. Si la probabilidad es baja, el sistema podría sugerir ajustes (ej. cambiar a un tono más emocional, publicar en otro horario, agregar video).

Guardamos el pipeline entrenado para poder reutilizarlo sin reentrenar:


In [ ]:
# Guardar el modelo/pipeline entrenado en Drive para reutilizarlo después
RUTA_MODELO = '/content/drive/MyDrive/colaba/modelo_viralidad_tecnonova'
modelo.write().overwrite().save(RUTA_MODELO)
print("Modelo guardado en:", RUTA_MODELO)

Modelo guardado en: /content/drive/MyDrive/colaba/modelo_viralidad_tecnonova


In [ ]:
# Para cargarlo más adelante (en otra sesión de Colab):
# from pyspark.ml import PipelineModel
# modelo_cargado = PipelineModel.load(RUTA_MODELO)

## Fase 6 — Monitoreo y mantenimiento (Monitoring & Maintenance)

Como parte de CRISP-ML(Q), el trabajo no termina al desplegar el modelo. En producción se recomendaría:

- **Monitorear el desempeño real**: comparar la predicción de viralidad contra el resultado real de cada publicación (¿realmente se volvió viral?) y recalcular AUC/accuracy periódicamente.
- **Detectar *data drift***: el lenguaje y las tendencias en redes sociales cambian rápido; el lexicón de sentimiento y el vocabulario del modelo (TF-IDF) pueden quedar desactualizados en pocos meses.
- **Reentrenar** el pipeline con datos nuevos de forma periódica (por ejemplo, mensualmente), incorporando publicaciones recientes y su resultado real de viralidad.
- **Ampliar el lexicón de sentimiento** o reemplazarlo por un modelo de NLP más robusto (ej. embeddings o un modelo preentrenado) si el volumen de datos crece lo suficiente como para justificarlo.


## Conclusiones

- El análisis de sentimiento basado en lexicón permitió etiquetar cada publicación como positiva, negativa o neutra de forma simple y explicable, usando únicamente transformaciones distribuidas de Spark (`udf`, `withColumn`).
- Los datos sugieren que la **carga emocional** del texto (ya sea positiva o negativa) está más asociada a la viralidad que el contenido neutro/informativo, lo cual tiene sentido de negocio: los extremos generan más interacción que la información plana.
- Apache Spark y MLlib permitieron construir, en un solo `Pipeline`, todo el flujo de preparación de datos (texto + categóricas + numéricas) y modelado, algo que sería mucho más difícil de mantener con herramientas tradicionales a medida que el dataset crece a los volúmenes reales del caso de negocio (millones de publicaciones).
- La metodología **CRISP-ML(Q)** ayudó a estructurar el proyecto más allá del modelo en sí: obligó a pensar en el objetivo de negocio, en cómo se desplegaría el modelo y en cómo se monitorearía en el tiempo, no solo en la métrica de accuracy.


## Publicar el proyecto en GitHub

Para cumplir con el requisito de documentar y subir el código a un repositorio de control de versiones, desde una terminal (no desde Colab) puedes ejecutar:

```bash
git init
git add Laboratorio06_Spark_Sentimiento_Viralidad.ipynb
git commit -m "Laboratorio 06: Apache Spark + CRISP-ML - sentimiento y viralidad TecnoNova"
git branch -M main
git remote add origin https://github.com/<tu-usuario>/<tu-repositorio>.git
git push -u origin main
```

También puedes descargar este cuaderno desde Colab (`Archivo > Descargar > Descargar .ipynb`) y subirlo directamente desde la interfaz web de GitHub si prefieres no usar la línea de comandos.


In [ ]:
# Cerrar la sesión de Spark al finalizar el trabajo
spark.stop()